In [1]:
import torch
import torch.fft as fft
import json
import numpy as np
import os
import torch.nn as nn
import clip

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from load_eegdatasets import SoloEEGDataset
from torch.utils.data import DataLoader, Dataset

# Load the configuration from the JSON file
config_path = "data_config.json"
with open(config_path, "r") as config_file:
    config = json.load(config_file)

# Access the paths from the config
data_path = config["data_path"]

train_dataset = SoloEEGDataset(data_path, subjects=['sub-08'], train=True)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, drop_last=True)

D:\Program\anaconda3\envs\BCI\Lib\site-packages\diffusers\utils\outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


eeg_data_tensor torch.Size([16540, 63, 100])
label_tensor torch.Size([16540])


D:\reproduction\BCI\EEG_Decoder\ReVIS\load_eegdatasets.py:83: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_features = torch.load(features_filename)


vae_img_features shape is : torch.Size([16540, 4, 64, 64])


In [6]:
with torch.no_grad():
    for batch_idx, (eeg_data, label, clip_text_features, clip_img_features, vae_img_features, text, img) in enumerate(train_loader):
        print(f'clip_text_features shape is {clip_text_features.shape}')
        break

clip_text_features shape is torch.Size([32, 768])


In [8]:
from transformers import CLIPTextModel
text_encoder = CLIPTextModel.from_pretrained("./stable-diffusion-v1-5/", subfolder="text_encoder")

In [9]:
directory = img_directory_test = config["img_directory_test"]
dirnames = [d for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d))]   
dirnames.sort()

texts = []

# load text data
text_folders = dirnames
for dir in text_folders:
    try:
        idx = dir.index('_')
        text_description = dir[idx + 1:]
    except ValueError:
        print(f"Skipped: {dir} due to no '_' found.")
        continue

    new_text_description = f"This picture is {text_description}"
    texts.append(new_text_description)

In [18]:
    def ClipTextencoder(texts):
        text_inputs = torch.cat([clip.tokenize(t) for t in texts])
        with torch.no_grad():
            text_features = text_encoder(text_inputs)
            text_features = text_features[0]

        clip_text_features = F.normalize(text_features, dim=-1).detach()
        return clip_text_features

In [20]:
from torch.nn import functional as F
text_features = ClipTextencoder(texts)

In [21]:
print(text_features.shape)

1
torch.Size([200, 77, 768])


In [23]:
for i in clip_text_features.children():
    print(i)

AttributeError: 'Tensor' object has no attribute 'children'

In [24]:
print(text_encoder)

CLIPTextModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), ep

In [4]:
def extract_id_from_string(s):
    match = re.search(r'\d+$', s)
    if match:
        return int(match.group())
    return None

In [ ]:
def train(model, sub, dataloader, optimizer, config, ):
    """ train one epoch """
    eeg_model.train()
    mse_loss_fn = nn.MSELoss()
    for batch_idx, batch in enumerate(train_loader):
        latents = batch["vae_img_features"]

In [ ]:
def train_loop(model, sub, unet):
    for epoch in range(config.num_train_epochs):

In [ ]:
def evaluate

In [ ]:
import torch.nn.functional as F
import json
from utils.Configs import Configs
import re
from accelerate import Accelerator
import itertools
from diffusers import DDPMScheduler, UNet2DConditionModel

from load_eegdatasets import SoloEEGDataset
from torch.utils.data import DataLoader, Dataset

from model.ReVisModels_Draft import ReVisEncoder

# Load the configuration from the JSON file
data_config_path = "data_config.json"
with open(data_config_path, "r") as data_config_file:
    data_config = json.load(data_config_file)
    
pretrained_model_path = data_config["pretrained_model_path"]
# Access the paths from the config
data_path = config["data_path"]
default_configs = Configs()

accelerator = Accelerator()

# def main
noise_scheduler = DDPMScheduler.from_pretrained(pretrained_model_path, subfolder="scheduler")
frozen_unet = UNet2DConditionModel.from_pretrained(pretrained_model_path, subfolder="unet")
frozen_unet.requires_grad_(False)
frozen_unet.to(accelerator.device)

# for sub in subjects:
sub = 'sub-08'

SemanticEncoder = ReVisEncoder(default_configs, encoder_type = 'semantic')
optimizer = torch.optim.AdamW(itertools.chain(SemanticEncoder.parameters()), lr=default_configs.learning_rate)
train_dataset = SoloEEGDataset(data_path, subjects=['sub-08'], train=True)
train_loader = DataLoader(train_dataset, batch_size=default_configs.train_batch_size, shuffle=True, num_workers=0, drop_last=True)

SemanticEncoder, optimizer, train_loader = accelerator.prepare(SemanticEncoder, optimizer, train_loader)
for epoch in range(default_configs.num_train_epochs):
    for idx, batch in enumerate(train_loader):
        with accelerator.accumulate(SemanticEncoder):
            latents = batch["vae_img_features"]
            noise = torch.randn_like(latents)
            bsz = latents.shape[0]
            subject_id = extract_id_from_string(sub)
            subject_ids = torch.full((bsz,), subject_id, dtype=torch.long).to(accelerator.device)
            
            # Sample a random timestep for each image
            timesteps = torch.randint(0, noise_scheduler.num_train_timesteps, (bsz,), device=latents.device)
            timesteps = timesteps.long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
            encoder_hidden_states = SemanticEncoder(batch["eeg_data"].to(accelerator.device), sub_ids = subject_ids)
            noise_pred = frozen_unet(noisy_latents, timesteps, encoder_hidden_states).sample
            loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")
            accelerator.backward(loss) 
            optimizer.step()
            optimizer.zero_grad()
            
            avg_loss = accelerator.gather(loss.repeat(bsz)).mean()
            if accelerator.is_main_process:
                print(f'loss is {avg_loss}')
